In [1]:
from IPython.display import display, HTML
display(HTML("<style>.container { width:98% !important; }</style>"))
%load_ext autoreload  
%autoreload 2
from IPython.core.interactiveshell import InteractiveShell
InteractiveShell.ast_node_interactivity = "all"

In [2]:
%matplotlib inline

In [3]:
import math
import numpy as np
import matplotlib.pyplot as plt
plt.style.use('ggplot')

In [4]:
import os
import time

from typing import Iterable
from dataclasses import dataclass

import numpy as np
import torch
import torch.nn as nn
import torch.optim as optim
import torch.nn.functional as F

from torchvision import datasets, transforms

from torch.optim import lr_scheduler

In [5]:
import sys 
import gc
for p in ['../..']:
    if p not in sys.path:
        # print(f"insert {p}")
        sys.path.insert(0, p)
print(sys.path)

['../..', '/home/kbardool/miniconda3/envs/opencv/lib/python312.zip', '/home/kbardool/miniconda3/envs/opencv/lib/python3.12', '/home/kbardool/miniconda3/envs/opencv/lib/python3.12/lib-dynload', '', '/home/kbardool/miniconda3/envs/opencv/lib/python3.12/site-packages', '/tmp/tmpr685z4uv']


In [6]:
from training_code import *
from wandb_helper import wandb_init, wandb_log_metrics,wandb_watch, wandb_end
from types import SimpleNamespace

In [7]:
from torchinfo import summary
from models import *
summary_col_names = ("input_size",
            "output_size",
            "num_params",
            # "params_percent",
            "kernel_size",
            "mult_adds",
            "trainable" )

In [8]:
from models_2 import *

In [9]:
input_size = (4, 3, 224, 224)

In [10]:
class Model_v00(nn.Module):
    def __init__(self, dropout=True):
        super().__init__()
        self.dropout = dropout
        if self.dropout:
            self.name = "Model_v00"
        else:
            self.name = "Model_v00_No-DO"
            
        # convolution layers
        self.body = nn.Sequential(
            nn.Conv2d(in_channels=3, out_channels=32, kernel_size=7),
            nn.ReLU(inplace=True),
            nn.MaxPool2d(kernel_size=2),
            
            nn.Conv2d(in_channels=32, out_channels=64, kernel_size=5),
            nn.ReLU(inplace=True),
            nn.MaxPool2d(kernel_size=2),

            nn.Flatten()
        )

        
        # Fully connected layers
        self.head = nn.Sequential(
            nn.Linear(in_features=64*52*52, out_features=1024), 
            nn.ReLU(inplace=True),
            
            nn.Linear(in_features=1024, out_features=3)
            
        )
    
    def forward(self, x):
        
        # apply feature extractor
        x = self.body(x)
        # apply classification head
        x = self.head(x)
        
        return x



In [52]:
# model = Model_v031() 
# model = Model_v81b() # MyModel = Model_v41
model = Model_v71b() # MyModel = Model_v41
print(model)

Model_v71b(
  (body): Sequential(
    (0): Conv2d(3, 64, kernel_size=(5, 5), stride=(1, 1), padding=(2, 2))
    (1): BatchNorm2d(64, eps=1e-05, momentum=0.1, affine=True, track_running_stats=True)
    (2): ReLU(inplace=True)
    (3): MaxPool2d(kernel_size=2, stride=2, padding=0, dilation=1, ceil_mode=False)
    (4): Conv2d(64, 128, kernel_size=(3, 3), stride=(1, 1), padding=(1, 1))
    (5): BatchNorm2d(128, eps=1e-05, momentum=0.1, affine=True, track_running_stats=True)
    (6): ReLU(inplace=True)
    (7): MaxPool2d(kernel_size=2, stride=2, padding=0, dilation=1, ceil_mode=False)
    (8): Conv2d(128, 256, kernel_size=(3, 3), stride=(1, 1), padding=(1, 1))
    (9): BatchNorm2d(256, eps=1e-05, momentum=0.1, affine=True, track_running_stats=True)
    (10): ReLU(inplace=True)
    (11): MaxPool2d(kernel_size=2, stride=2, padding=0, dilation=1, ceil_mode=False)
    (12): Conv2d(256, 512, kernel_size=(3, 3), stride=(1, 1), padding=(1, 1))
    (13): BatchNorm2d(512, eps=1e-05, momentum=0.1, af

In [53]:
print(f"\n {model.name}")
print(summary(model.body, input_size = input_size, col_names=summary_col_names))
print(summary(model, input_size = input_size, col_names=summary_col_names))


 Model_v71b
Layer (type:depth-idx)                   Input Shape               Output Shape              Param #                   Kernel Shape              Mult-Adds                 Trainable
Sequential                               [4, 3, 224, 224]          [4, 4096]                 --                        --                        --                        True
├─Conv2d: 1-1                            [4, 3, 224, 224]          [4, 64, 224, 224]         4,864                     [5, 5]                    976,224,256               True
├─BatchNorm2d: 1-2                       [4, 64, 224, 224]         [4, 64, 224, 224]         128                       --                        512                       True
├─ReLU: 1-3                              [4, 64, 224, 224]         [4, 64, 224, 224]         --                        --                        --                        --
├─MaxPool2d: 1-4                         [4, 64, 224, 224]         [4, 64, 112, 112]         --         

In [50]:
for  mdl in ['Model_v06', 'Model_v61', 'Model_v61a', 'Model_v61b', 'Model_v62', 'Model_v62a', 'Model_v71','Model_v71b', 'Model_v71c', 'Model_v81b']:
    if mdl in globals():
        model = globals()[mdl]()
        summ = summary(model, input_size = input_size, col_names=summary_col_names)
        # print(type(summ))
        print(f"Model: {mdl:11s}  total_params: {summ.total_params:14,d}     trainable_params: {summ.trainable_params:14,d}")

Model: Model_v06    total_params:     38,400,899     trainable_params: 38,400,899
Model: Model_v61    total_params:     88,738,691     trainable_params: 88,738,691
Model: Model_v61a   total_params:     88,738,435     trainable_params: 88,738,435
Model: Model_v61b   total_params:     88,738,819     trainable_params: 88,738,819
Model: Model_v62    total_params:     65,010,051     trainable_params: 65,010,051
Model: Model_v62a   total_params:     65,009,795     trainable_params: 65,009,795
Model: Model_v71    total_params:    111,159,683     trainable_params: 111,159,683
Model: Model_v71b   total_params:    111,159,811     trainable_params: 111,159,811
Model: Model_v71c   total_params:    111,159,811     trainable_params: 111,159,811
Model: Model_v81b   total_params:    137,352,451     trainable_params: 137,352,451


In [32]:
summ.__dict__
print(summ)

{'summary_list': [Model_v61: 0,
  Sequential: 1,
  Conv2d: 2,
  ReLU: 2,
  MaxPool2d: 2,
  Conv2d: 2,
  BatchNorm2d: 2,
  ReLU: 2,
  MaxPool2d: 2,
  Conv2d: 2,
  BatchNorm2d: 2,
  ReLU: 2,
  MaxPool2d: 2,
  Conv2d: 2,
  BatchNorm2d: 2,
  ReLU: 2,
  MaxPool2d: 2,
  Conv2d: 2,
  BatchNorm2d: 2,
  ReLU: 2,
  MaxPool2d: 2,
  Conv2d: 2,
  BatchNorm2d: 2,
  ReLU: 2,
  MaxPool2d: 2,
  Flatten: 2,
  Sequential: 1,
  Linear: 2,
  ReLU: 2,
  Linear: 2,
  ReLU: 2,
  Dropout: 2,
  Linear: 2],
 'input_size': [(4, 3, 224, 224)],
 'formatting': <torchinfo.formatting.FormattingOptions at 0x772cb8d8b380>,
 'total_input': 2408520,
 'total_mult_adds': 48335928332,
 'total_params': 88738691,
 'trainable_params': 88738691,
 'total_param_bytes': 354954764,
 'total_output_bytes': 298811488}

Layer (type:depth-idx)                   Input Shape               Output Shape              Param #                   Kernel Shape              Mult-Adds                 Trainable
Model_v61                                [4, 3, 224, 224]          [4, 3]                    --                        --                        --                        True
├─Sequential: 1-1                        [4, 3, 224, 224]          [4, 8192]                 --                        --                        --                        True
│    └─Conv2d: 2-1                       [4, 3, 224, 224]          [4, 64, 224, 224]         9,472                     [7, 7]                    1,901,068,288             True
│    └─ReLU: 2-2                         [4, 64, 224, 224]         [4, 64, 224, 224]         --                        --                        --                        --
│    └─MaxPool2d: 2-3                    [4, 64, 224, 224]         [4, 64, 112, 112]         --                      

In [25]:
print(f"\n {model.name}")
print(summary(model, input_size = input_size, col_names=summary_col_names))


 Model_v81b
Layer (type:depth-idx)                   Input Shape               Output Shape              Param #                   Kernel Shape              Mult-Adds                 Trainable
Model_v81b                               [4, 3, 224, 224]          [4, 3]                    --                        --                        --                        True
├─Sequential: 1-1                        [4, 3, 224, 224]          [4, 4096]                 --                        --                        --                        True
│    └─Conv2d: 2-1                       [4, 3, 224, 224]          [4, 256, 224, 224]        19,456                    [5, 5]                    3,904,897,024             True
│    └─BatchNorm2d: 2-2                  [4, 256, 224, 224]        [4, 256, 224, 224]        512                       --                        2,048                     True
│    └─ReLU: 2-3                         [4, 256, 224, 224]        [4, 256, 224, 224]        --       

In [13]:
# model.__dict__
model.modules

<bound method Module.modules of Model_v061a(
  (body): Sequential(
    (0): Conv2d(3, 64, kernel_size=(7, 7), stride=(1, 1), padding=(3, 3))
    (1): ReLU(inplace=True)
    (2): MaxPool2d(kernel_size=2, stride=2, padding=0, dilation=1, ceil_mode=False)
    (3): Conv2d(64, 128, kernel_size=(5, 5), stride=(1, 1), padding=(2, 2))
    (4): ReLU(inplace=True)
    (5): MaxPool2d(kernel_size=2, stride=2, padding=0, dilation=1, ceil_mode=False)
    (6): Conv2d(128, 256, kernel_size=(5, 5), stride=(1, 1), padding=(2, 2))
    (7): BatchNorm2d(256, eps=1e-05, momentum=0.1, affine=True, track_running_stats=True)
    (8): ReLU(inplace=True)
    (9): MaxPool2d(kernel_size=2, stride=2, padding=0, dilation=1, ceil_mode=False)
    (10): Conv2d(256, 512, kernel_size=(5, 5), stride=(1, 1), padding=(2, 2))
    (11): BatchNorm2d(512, eps=1e-05, momentum=0.1, affine=True, track_running_stats=True)
    (12): ReLU(inplace=True)
    (13): MaxPool2d(kernel_size=2, stride=2, padding=0, dilation=1, ceil_mode=Fals

In [ ]:
isinstance(kevin, nn.Module)

In [28]:
kevin.__class__.__name__

'Model_v00'

In [60]:
import inspect
p = inspect.stack()
for pp in p:
    print(pp.function)

<module>
run_code
run_ast_nodes
run_cell_async
_pseudo_sync_runner
_run_cell
run_cell
run_cell
do_execute
execute_request
execute_request
dispatch_shell
shell_main
_run
_run_once
run_forever
start
start
launch_instance
<module>
_run_code
_run_module_as_main


In [88]:
from mytorchinfo import summary
summary(kevin)

 Summary Main Routine
 Validate User Params
 Process Input
 Forward Pass for Model_v00
 Apply hooks for Model_v00 
 init LayerInfo: var_name: Model_v00  name: Model_v00  layer_id: 133654031722272  [ caller forward_pass ---> apply_hooks ---> __init__ ]
 Construct_pre_hook for Model_v00 at depth 0 
 Call prehook for Model_v00
 init LayerInfo: var_name: Model_v00  name: Model_v00  layer_id: 133654031722272  [ caller apply_hooks ---> pre_hook ---> __init__ ]
 pre_hook for . . . 133654031722272  Model_v00  0
 Apply hooks for body 
 init LayerInfo: var_name: body  name: Sequential  layer_id: 133654031722176  [ caller forward_pass ---> apply_hooks ---> __init__ ]
 Construct_pre_hook for body at depth 1 
 Call prehook for body
 init LayerInfo: var_name: body  name: Sequential  layer_id: 133654031722176  [ caller apply_hooks ---> pre_hook ---> __init__ ]
 pre_hook for . . . 133654031722176  body  1
 Apply hooks for 0 
 init LayerInfo: var_name: 0  name: Conv2d  layer_id: 133654031723616  [ call

Layer (type:depth-idx)                   Param #
Model_v00                                --
├─Sequential: 1-1                        --
│    └─Conv2d: 2-1                       4,736
│    └─ReLU: 2-2                         --
│    └─MaxPool2d: 2-3                    --
│    └─Conv2d: 2-4                       51,264
│    └─ReLU: 2-5                         --
│    └─MaxPool2d: 2-6                    --
│    └─Flatten: 2-7                      --
├─Sequential: 1-2                        --
│    └─Linear: 2-8                       177,210,368
│    └─ReLU: 2-9                         --
│    └─Linear: 2-10                      3,075
Total params: 177,269,443
Trainable params: 177,269,443
Non-trainable params: 0

In [77]:
for name, mod in reversed(kevin._modules.items()):
    print(f" name: {name}   mod: {mod}")

 name: head   mod: Sequential(
  (0): Linear(in_features=173056, out_features=1024, bias=True)
  (1): ReLU(inplace=True)
  (2): Linear(in_features=1024, out_features=3, bias=True)
)
 name: body   mod: Sequential(
  (0): Conv2d(3, 32, kernel_size=(7, 7), stride=(1, 1))
  (1): ReLU(inplace=True)
  (2): MaxPool2d(kernel_size=2, stride=2, padding=0, dilation=1, ceil_mode=False)
  (3): Conv2d(32, 64, kernel_size=(5, 5), stride=(1, 1))
  (4): ReLU(inplace=True)
  (5): MaxPool2d(kernel_size=2, stride=2, padding=0, dilation=1, ceil_mode=False)
  (6): Flatten(start_dim=1, end_dim=-1)
)


In [72]:
kevin.modules

TypeError: Module.modules() takes 1 positional argument but 2 were given

In [34]:
from __future__ import annotations

from typing import Any, Dict, Iterable, Sequence, Union

import numpy as np
import torch
from torch import nn
from torch.jit import ScriptModule

from torchinfo.enums import ColumnSettings

try:
    from torch.nn.parameter import is_lazy
except ImportError:

    def is_lazy(param: nn.Parameter) -> bool:  # type: ignore[misc]
        del param
        return False

DETECTED_INPUT_OUTPUT_TYPES = Union[
    Sequence[Any], Dict[Any, torch.Tensor], torch.Tensor
]